# Word2Vec — Word Embeddings

**Course:** Artificial Intelligence · Unit 6 — Language and NLP  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Russell, S. & Norvig, P. (2022). *Artificial Intelligence: A Modern Approach* (4th ed.), Ch. 24 — Natural Language Processing  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Train a **Word2Vec Skip-gram** model on a Spanish tweet corpus
2. Retrieve the **most similar words** to a query using cosine similarity
3. Perform **word analogy** reasoning (king – man + woman ≈ queen)
4. Inspect individual **word vectors** and their dimensionality
5. Visualise word embeddings in 2D using **PCA**
6. Load pre-trained embeddings from `gensim.downloader`

## What is Word2Vec?

Word2Vec (Mikolov et al., 2013) learns dense vector representations (embeddings) by training a neural network to predict a word from its context (CBOW) or to predict context from a word (Skip-gram). Semantically similar words end up close in the embedding space.


---

## 1. Environment Setup

Install dependencies **once** in your terminal:

```bash
pip install gensim spacy scikit-learn matplotlib pandas
python -m spacy download es_core_news_sm
```


In [ ]:
# pip install gensim spacy scikit-learn matplotlib pandas  # run once in terminal
# python -m spacy download es_core_news_sm                 # run once in terminal
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spacy
from gensim.models import Word2Vec
from sklearn.decomposition import PCA

DIR_DATA = Path.cwd() / 'data'


---

## 2. Load spaCy Spanish Model


In [ ]:
try:
    nlp = spacy.load('es_core_news_sm')
    print(f'Model loaded: {nlp.meta["name"]} v{nlp.meta["version"]}')
except OSError:
    raise SystemExit('Run: python -m spacy download es_core_news_sm')


---

## 3. Load Training Data

The TASS tweet corpus (`TASS2018.csv`) is used as training data. The `Tweet` column contains raw Spanish tweet text.


In [ ]:
# Load the TASS 2018 CSV file (semicolon-separated)
data_train = pd.read_csv(DIR_DATA / 'TASS2018.csv', sep=';')

print(f'Dataset shape: {data_train.shape}')
print(f'Columns: {data_train.columns.tolist()}')
print(data_train.head(3).to_string())


> **Output interpretation:** The dataset contains Spanish tweets with associated metadata (date, polarity, content). The `Tweet` or `content` column provides the raw text that will be tokenised and fed to Word2Vec. More training data generally improves embedding quality — Word2Vec typically needs millions of tokens to learn reliable representations.


---

## 4. Text Pre-processing and Tokenisation

Before training, each tweet is normalised (remove URLs, mentions, special characters) and tokenised with spaCy. Tokens shorter than 2 characters and non-alphabetic tokens are discarded.


In [ ]:
def remove_noise(text: str) -> str:
    """Remove URLs, user mentions, hashtag symbols, and non-alphanumeric characters.

    Args:
        text: Raw tweet string.

    Returns:
        Cleaned text with only alphanumeric characters and spaces.
    """
    text = re.sub(r'http\S+|www\S+', '', text)   # remove URLs
    text = re.sub(r'@\w+', '', text)              # remove @mentions
    text = re.sub(r'[^\w\s]', '', text)           # remove punctuation
    return text.strip()


def tokenize_tweet(text: str) -> list:
    """Tokenise a Spanish tweet using spaCy.

    Args:
        text: Pre-cleaned tweet string.

    Returns:
        List of lowercase alphabetic token strings (length >= 2).
    """
    doc = nlp(remove_noise(text))
    return [
        token.text.lower()
        for token in doc
        if token.is_alpha and len(token.text) >= 2
    ]


# Apply tokenisation to the full corpus
tweet_col = 'Tweet' if 'Tweet' in data_train.columns else 'content'
data = pd.DataFrame({'tokens': data_train[tweet_col].apply(tokenize_tweet)})

print(f'Total tokenised tweets: {len(data):,}')
print(f'Sample tokens (tweet 0): {data["tokens"].iloc[0]}')


> **Output interpretation:** Each row in `data` is now a list of clean lowercase tokens. Removing URLs, mentions, and punctuation prevents the model from learning spurious co-occurrences between usernames and sentiment words. spaCy's tokeniser handles Spanish contractions and clitics correctly.


---

## 5. Train Word2Vec Skip-gram Model

| Parameter | Value | Meaning |
|---|---|---|
| `vector_size` | 100 | Dimensionality of each word embedding |
| `window` | 5 | Context window: 5 words to each side |
| `min_count` | 1 | Keep all tokens (raise to 5+ for large corpora) |
| `workers` | 4 | Parallel training threads |
| `sg` | 1 | Skip-gram architecture (0 = CBOW) |


In [ ]:
sentences = data['tokens'].tolist()

# Initialise and train the Skip-gram model
model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,          # 1 = Skip-gram; 0 = CBOW
    epochs=10,
)

print(f'Vocabulary size: {len(model.wv.key_to_index):,} words')
print(f'Embedding dimension: {model.vector_size}')


> **Output interpretation:** The vocabulary size reflects the number of unique tokens seen at least `min_count` times. With `min_count=1`, every token is kept — appropriate for small corpora like TASS (~5,000 tweets). For larger corpora, set `min_count=5` to discard rare words that lack enough co-occurrence data to learn reliable embeddings.


---

## 6. Semantic Similarity and Word Analogies


In [ ]:
# Find the 5 words most similar to a query word
query_word = 'mujer'  # 'woman' in Spanish
if query_word in model.wv:
    similar = model.wv.most_similar(query_word, topn=5)
    print(f'Words most similar to "{query_word}":')
    for word, score in similar:
        print(f'  {word:20s} cosine similarity = {score:.4f}')
else:
    print(f'Word "{query_word}" not in vocabulary.')


> **Output interpretation:** Cosine similarity ranges from −1 (opposite meaning) to +1 (identical direction in embedding space). Words with similarity > 0.7 are semantically close. On a small corpus the results may not match intuition — train on a larger dataset (e.g., CC-100 Spanish Wikipedia) for better quality embeddings.


In [ ]:
# Word analogy: hombre - rey + mujer ≈ ? (man - king + woman ≈ queen)
try:
    analogy = model.wv.most_similar(
        positive=['mujer', 'rey'],   # woman + king
        negative=['hombre'],          # - man
        topn=3,
    )
    print('Analogy: mujer + rey - hombre ≈ ?')
    for word, score in analogy:
        print(f'  {word:20s} score = {score:.4f}')
except KeyError as e:
    print(f'Word not in vocabulary: {e}')


> **Output interpretation:** Word analogies exploit the linear structure of the embedding space. On a large balanced corpus the classic result is `king – man + woman ≈ queen`. On small domain-specific corpora (tweets) the analogy may not hold because the training signal is insufficient — this is expected and illustrates why pre-trained embeddings (FastText, BERT) are preferred in practice.


---

## 7. Inspect a Word Vector


In [ ]:
target_word = 'hombre'  # 'man' in Spanish
if target_word in model.wv:
    vec = model.wv[target_word]
    print(f'Word: "{target_word}"')
    print(f'Vector shape: {vec.shape}')
    print(f'First 10 dimensions: {vec[:10].round(4)}')
    print(f'L2 norm: {np.linalg.norm(vec):.4f}')
else:
    print(f'Word "{target_word}" not in vocabulary.')


> **Output interpretation:** The embedding is a 100-dimensional float vector. Individual dimensions have no interpretable meaning — meaning emerges from the geometric relationships between vectors (cosine similarity, analogies). The L2 norm is not normalised by default in gensim; use `model.wv.get_normed_vectors()` to work with unit-length vectors.


---

## 8. Visualise Embeddings with PCA

PCA reduces the 100-dimensional embedding space to 2D for plotting. The relative distances are approximately preserved — semantically similar words cluster together.


In [ ]:
# Select the top-N most frequent words for visualisation
N_WORDS = 50
vocab_words = list(model.wv.key_to_index.keys())[:N_WORDS]
word_vectors = np.array([model.wv[w] for w in vocab_words])

# Reduce to 2D with PCA
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(word_vectors)

fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(coords[:, 0], coords[:, 1], alpha=0.4, s=30)
for i, word in enumerate(vocab_words):
    ax.annotate(word, coords[i], fontsize=8, alpha=0.8)
ax.set_title(f'Word2Vec Skip-gram Embeddings — PCA (top {N_WORDS} words)')
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
plt.tight_layout()
plt.show()


> **Output interpretation:** Semantically related words should cluster near each other in the 2D projection. The PCA explains only a small fraction of the total variance (100D → 2D loses most information), so the visualisation is an approximation. Use t-SNE or UMAP for better preservation of local neighbourhoods in high-dimensional embedding spaces.


---

## 9. Load Pre-trained Embeddings from Gensim Hub

Gensim provides pre-trained embeddings trained on large corpora. These are far better than embeddings trained on a small tweet dataset.


In [ ]:
import gensim.downloader

# List available pre-trained models
available = list(gensim.downloader.info()['models'].keys())
print('Available pre-trained models (first 10):')
print(available[:10])

# Example: load GloVe Twitter embeddings (100d)
# glove = gensim.downloader.load('glove-twitter-100')  # ~400 MB — uncomment to download
# print(f'GloVe vocab size: {len(glove.key_to_index):,}')


> **Output interpretation:** Pre-trained embeddings like GloVe (trained on billions of tweets or Wikipedia tokens) capture far richer semantic relationships than embeddings trained on a few thousand TASS tweets. For production Spanish NLP, consider `fasttext-wiki-news-subwords-300` (supports subword morphology, handles unknown words) or a multilingual BERT variant.
